In [44]:
import copy
import pandas as pd

pd.set_option('display.expand_frame_repr', False)   # to show all cols in one line

holidays = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/holidays_events.csv')
stores = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/stores.csv')

## 1. Holidays

In [43]:
print(f'shape = {holidays.shape}')
print(holidays.dtypes)

shape = (350, 6)
date            str
type            str
locale          str
locale_name     str
description     str
transferred    bool
dtype: object


In [36]:
holidays.head()

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


In [38]:
holidays.describe()

,date,type,locale,locale_name,description,transferred
count,350,350,350,350,350,350
unique,312,6,3,24,103,2
top,2014-06-25,Holiday,National,Ecuador,Carnaval,False
freq,4,221,174,174,10,338


### Preprocessing

--the crucial information for us is whether a given day is a holiday or not in a given location

--add column `holiday` with True as default value to be overwritten to False later if needed

In [49]:
holidays_processed = copy.deepcopy(holidays)
holidays_processed['holiday'] = True

### - type

> Days that are type `Bridge` are extra days that are added to a holiday (e.g., to extend the break across a long weekend). These are frequently made up by the type `Work Day` which is a day not normally scheduled for work (e.g., Saturday) that is meant to payback the Bridge.

> Type `Additional` holidays are days added a regular calendar holiday, for example, as typically happens around Christmas (making Christmas Eve a holiday).

==> treat `Bridge` as holiday and `Work Day` as working day

==> `Additional` may be an indicator for holidays before which sales go up

In [51]:
mask_wd = holidays_processed["type"].eq("Work Day")
holidays_processed.loc[mask_wd, "holiday"] = False

### - transferred

> A holiday that is `transferred` officially falls on that calendar day, but was moved to another date by the government. A transferred day is more like a normal day than a holiday. To find the day that it was actually celebrated, look for the corresponding row where type is `Transfer`.

==> treat as working day

In [54]:
mask_tf = holidays_processed['transferred'].eq(True)
holidays_processed.loc[mask_tf, 'holiday'] = False

### - description

possibly irrelevant, but leaving it for now

### - locale / locale_name

taken together, will provide the information **per store** whether a given day is a holiday

(implicational hierarchy: country > state > city)

### - check for duplicate rows

In [61]:
# wrt 'date'
dup_date = holidays_processed[holidays_processed.duplicated(subset = ['date'], keep=False)].sort_values(by = ['date'])
dup_date

,date,type,locale,...,description,transferred,holiday
7,2012-06-25,Holiday,Regional,...,Provincializacion de Imbabura,False,True
8,2012-06-25,Holiday,Local,...,Cantonizacion de Latacunga,False,True
9,2012-06-25,Holiday,Local,...,Fundacion de Machala,False,True
10,2012-07-03,Holiday,Local,...,Fundacion de Santo Domingo,False,True
11,2012-07-03,Holiday,Local,...,Cantonizacion de El Carmen,False,True
...,...,...,...,...,...,...,...
319,2017-07-03,Holiday,Local,...,Fundacion de Santo Domingo,False,True
341,2017-12-08,Holiday,Local,...,Fundacion de Loja,False,True
342,2017-12-08,Transfer,Local,...,Traslado Fundacion de Quito,False,True
344,2017-12-22,Holiday,Local,...,Cantonizacion de Salinas,False,True


In [65]:
# or e.g. wrt date and locale_name
dup_date_ln = holidays_processed[holidays_processed.duplicated(subset = ['date', 'locale_name'], keep=False)].sort_values(by = ['date'])
dup_date_ln

,date,type,locale,...,description,transferred,holiday
35,2012-12-24,Bridge,National,...,Puente Navidad,False,True
36,2012-12-24,Additional,National,...,Navidad-1,False,True
39,2012-12-31,Bridge,National,...,Puente Primer dia del ano,False,True
40,2012-12-31,Additional,National,...,Primer dia del ano-1,False,True
156,2014-12-26,Bridge,National,...,Puente Navidad,False,True
157,2014-12-26,Additional,National,...,Navidad+1,False,True
235,2016-05-01,Holiday,National,...,Dia del Trabajo,False,True
236,2016-05-01,Event,National,...,Terremoto Manabi+15,False,True
242,2016-05-07,Additional,National,...,Dia de la Madre-1,False,True
243,2016-05-07,Event,National,...,Terremoto Manabi+21,False,True


-only some of them are real duplicates

-no conflicting information in `holiday`

-the information in `type` may still turn out to be useful

==> probably no need to bother to clean them

## 2. Stores